# SereBench — Paper Plots (2nd campaign, decoupled evaluation)

This notebook regenerates the paper figures for the **redesigned experimental campaign**, in which evaluation is fully decoupled from training:

* **HSJA** (clean anchors) is the *primary*, training-independent robustness metric.
* a **fixed Gaussian sigma-grid** (`adv_eval/sigma_<s>/*`), identical for every vehicle, is the *secondary* robustness profile.

New W&B group names:

* Block A (ET1/ET2, no FL): `et2-noadv-nofl`, `et2-adv-nofl`
* Block B (ET3, FL transfer): `et3-freerider-{nofl,fedavg,fedprox,fedyogi,fedmedian}`
* architectures add `-cnn` / `-resnet` (mlp = no suffix).

Figures: **ET1/ET2** (HSJA 3x6 grid, unchanged proposal), **ET3** (FL robustness transfer to the clean free-rider Angela — new proposals), **ET4** (sigma-degradation curves exploiting the new grid), and **Fig 5** (representation-space grid).

## 1 — Imports & typography

In [ ]:
!pip install --upgrade wandb

In [ ]:
import os, re, itertools, warnings, pickle
from pathlib import Path
from math import ceil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from dotenv import load_dotenv
from scipy import stats as scipy_stats
import wandb

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

title_font  = {'weight': 'bold', 'size': 22}
axis_font   = {'weight': 'bold', 'size': 18}
legend_font = {'weight': 'bold', 'size': 16}
plt.rcParams['axes.linewidth'] = 1.6
plt.rcParams['figure.facecolor'] = 'white'
print('Imports OK')

## 2 — Catalogue (redesigned campaign)

In [ ]:
# Fleet. In Block A the canonical training-noise gradient still applies; in
# Block B all peers train at 1.0 and Angela trains clean (free-rider).
VEHICLES = ['angela', 'bob', 'claude', 'daniel']
VEHICLE_NOISE = {'angela': 0.0, 'bob': 0.7, 'claude': 1.4, 'daniel': 2.1}
FREE_RIDER = 'angela'
PEERS = ['bob', 'claude', 'daniel']

MODELS = ['mlp', 'cnn', 'resnet']

# Block A (ET1/ET2, no FL)
ET2_NOADV = 'et2-noadv-nofl'
ET2_ADV   = 'et2-adv-nofl'

# Block B (ET3, FL transfer, clean free-rider)
ET3_OFF = 'et3-freerider-nofl'
ET3_ON  = {'FedAvg':   'et3-freerider-fedavg',
           'FedProx':  'et3-freerider-fedprox',
           'FedYogi':  'et3-freerider-fedyogi',
           #'FedMedian':'et3-freerider-fedmedian'
           }

BASE_EXPERIMENTS = [ET2_NOADV, ET2_ADV, ET3_OFF] + list(ET3_ON.values())

DEFAULT_SEEDS = [42, 123, 456, 789, 1234]

# Fixed Gaussian sigma-grid (must match default_consumer_config.eval_sigmas).
EVAL_SIGMAS = [0.5, 1.0, 1.5, 2.0]
def sigma_tag(s):
    # Mirrors the consumer: f'{sigma:g}'.replace('.', '_')  -> 0.5->'0_5', 1.0->'1'
    return ('%g' % s).replace('.', '_')
def sigma_key(metric_base, s):
    # metric_base in {accuracy, precision, recall, f1, macro_f1}
    return 'adv_eval/sigma_%s/%s' % (sigma_tag(s), metric_base)

HSJA_METRICS = ['hsja_adv_eval/accuracy', 'hsja_adv_eval/precision',
                'hsja_adv_eval/recall', 'hsja_adv_eval/f1', 'hsja_adv_eval/macro_f1',
                'hsja_adv_eval/avg_perturbation', 'hsja_adv_eval/avg_queries']
ONLINE_METRICS = ['online_class_accuracy', 'online_class_precision',
                  'online_class_recall', 'online_class_f1', 'online_class_macro_f1']

def metric_key(vehicle, suffix):
    return '%s_statistics.%s' % (vehicle, suffix)

def with_arch(group, arch):
    return group if arch == 'mlp' else '%s-%s' % (group, arch)

_PRETTY = {
    'hsja_adv_eval/accuracy': 'Accuracy', 'hsja_adv_eval/precision': 'Precision',
    'hsja_adv_eval/recall': 'Recall', 'hsja_adv_eval/f1': 'F1 (weighted)',
    'hsja_adv_eval/macro_f1': 'Macro-F1',
    'hsja_adv_eval/avg_perturbation': 'Avg L2 perturbation',
    'hsja_adv_eval/avg_queries': 'Avg queries',
    'class_accuracy': 'Accuracy (train)', 'class_macro_f1': 'Macro-F1 (train)',
    'class_f1': 'F1 weighted (train)',
    'online_class_accuracy': 'Online Accuracy', 'online_class_macro_f1': 'Online Macro-F1',
}
def pretty(suffix):
    return _PRETTY.get(suffix, suffix.replace('_', ' ').title())

def ylim_for(suffix):
    s = suffix.lower()
    if any(t in s for t in ['accuracy', 'precision', 'recall', 'f1']):
        return (0, 1.02)
    if any(t in s for t in ['loss', 'perturbation', 'queries', 'time', 'processed']):
        return (0, None)
    return None

print('Catalogue ready:', len(BASE_EXPERIMENTS), 'base groups,', len(MODELS),
      'architectures,', len(VEHICLES), 'vehicles, sigmas =', EVAL_SIGMAS)

## 3 — W&B API & history fetch

In [ ]:
load_dotenv()
for parent in [Path.cwd()] + list(Path.cwd().parents):
    cand = parent / '.env'
    if cand.exists():
        load_dotenv(cand, override=False)
        break

WANDB_ENTITY       = os.getenv('WANDB_ENTITY', 'jfcevallos')
WANDB_PROJECT_NAME = os.getenv('WANDB_PROJECT_NAME', 'SereBench')
WANDB_PROJECT      = '%s/%s' % (WANDB_ENTITY, WANDB_PROJECT_NAME)

HISTORY_SAMPLES = 100_000
CACHE_PATH      = Path('_serebench2_history_cache.pkl')
FORCE_REFETCH   = False
# Restrict to the redesigned-campaign groups (None = all groups in the project).
GROUPS_FILTER   = set(BASE_EXPERIMENTS) | {with_arch(g, m) for g in BASE_EXPERIMENTS for m in MODELS}
# Optionally tag the new runs and filter on it (e.g. ['campaign2']); None = no tag filter.
WANDB_TAG_FILTER = ['NOZOMI_RUNS']

def _seed_of(run):
    m = re.search(r'_seed(\d+)', run.name or '')
    if m:
        return int(m.group(1))
    cfg = run.config or {}
    for path in (('default_consumer_config', 'seed'), ('default_vehicle_config', 'seed')):
        d = cfg
        try:
            for p in path:
                d = d[p]
            return int(d)
        except Exception:
            pass
    return -1

def fetch_runs():
    api = wandb.Api(timeout=60)
    runs = api.runs(WANDB_PROJECT)
    out = {}
    print('Scanning runs in', WANDB_PROJECT, '...')
    for run in runs:
        grp = run.group
        if grp is None:
            continue
        if GROUPS_FILTER and grp not in GROUPS_FILTER:
            continue
        if WANDB_TAG_FILTER and not any(tag in run.tags for tag in WANDB_TAG_FILTER):
            continue
        seed = _seed_of(run)
        try:
            hist = run.history(samples=HISTORY_SAMPLES)
        except Exception as exc:
            print('  ! history failed for', run.name, ':', exc)
            continue
        if '_step' in hist.columns:
            hist = hist.set_index('_step')
        prev = out.get((grp, seed))
        if prev is None or len(hist) >= len(prev):
            out[(grp, seed)] = hist
        print('  %-28s seed=%-5s steps=%6d  (%s)' % (grp, seed, len(hist), run.id))
    return out

if (not FORCE_REFETCH) and CACHE_PATH.exists():
    print('Loading cached histories from', CACHE_PATH)
    with open(CACHE_PATH, 'rb') as fh:
        RUNS = pickle.load(fh)
else:
    RUNS = fetch_runs()
    with open(CACHE_PATH, 'wb') as fh:
        pickle.dump(RUNS, fh)
    print('Cached to', CACHE_PATH)

print('\nTotal (group, seed) run-histories loaded:', len(RUNS))

## 4 — Discovery: which groups & sigma keys are present?

In [ ]:
GROUPS_PRESENT = sorted({g for (g, s) in RUNS})
print('Groups present (%d):' % len(GROUPS_PRESENT))
for g in GROUPS_PRESENT:
    seeds = sorted({s for (gg, s) in RUNS if gg == g})
    print('  %-30s seeds=%s' % (g, seeds))

# Sanity: confirm the new sigma-grid columns actually arrived in the histories.
_seen = set()
for hist in RUNS.values():
    for col in hist.columns:
        if '/sigma_' in col:
            _seen.add(col.split('.', 1)[1] if '.' in col else col)
print('\nSigma-grid metric columns seen (sample):')
for c in sorted(_seen)[:20]:
    print('  ', c)
if not _seen:
    print('  (none yet — runs from the new campaign may not be logged)')

## 5 — Plotting engine (CI-aware)

In [ ]:
def seed_series_for_group(group, key, smooth=1):
    cols = {}
    for (g, seed), hist in RUNS.items():
        if g != group or key not in hist.columns:
            continue
        s = pd.to_numeric(hist[key], errors='coerce').dropna().reset_index(drop=True)
        if smooth > 1:
            s = s.rolling(smooth, min_periods=1).mean()
        if len(s):
            cols['seed%s' % seed] = s
    if not cols:
        return None
    return pd.DataFrame(cols)

def compute_ci_band(wide, alpha=0.95):
    if wide is None or wide.shape[1] == 0:
        return None, None, None
    w = wide.apply(pd.to_numeric, errors='coerce')
    mean = w.mean(axis=1)
    std  = w.std(axis=1, ddof=1)
    n    = w.notna().sum(axis=1)
    t_mult = pd.Series(
        [scipy_stats.t.ppf((1 + alpha) / 2, df=max(int(ni) - 1, 1)) if ni > 1 else np.nan
         for ni in n], index=mean.index)
    margin = t_mult * std / np.sqrt(n.replace(0, np.nan))
    return mean, mean - margin, mean + margin

def converged_scalar(group, key, tail=5, smooth=2):
    # One converged value per seed = mean of the last `tail` benchmark rounds,
    # then mean +/- 95%% CI across seeds. Returns (mean, ci_margin, n_seeds).
    wide = seed_series_for_group(group, key, smooth)
    if wide is None or wide.empty:
        return np.nan, 0.0, 0
    finals = wide.tail(tail).mean(axis=0).dropna().values
    if len(finals) == 0:
        return np.nan, 0.0, 0
    m = float(np.mean(finals))
    if len(finals) > 1:
        se = np.std(finals, ddof=1) / np.sqrt(len(finals))
        margin = float(scipy_stats.t.ppf(0.975, df=len(finals) - 1) * se)
    else:
        margin = 0.0
    return m, margin, len(finals)

_HARDCODED = {
    'no-adv-training': '#E8743B', 'adv-training': '#19A979',
    'FL off': '#B06B5A', 'FL on': '#2C7FB8',
    'FedAvg': '#3F51B5', 'FedProx': '#009688', 'FedYogi': '#E91E63', 'FedMedian': '#FF9800',
    'mlp': '#2196F3', 'cnn': '#FF9800', 'resnet': '#4CAF50',
    'Angela': '#2E86AB', 'Bob': '#19A979', 'Claude': '#E8743B', 'Daniel': '#A23B72',
    'angela': '#2E86AB', 'bob': '#19A979', 'claude': '#E8743B', 'daniel': '#A23B72',
    'peers': '#19A979',
}
_fallback = itertools.cycle(plt.colormaps.get_cmap('tab10').colors)
_color_cache = {}
def color_for(label):
    if label in _HARDCODED:
        return _HARDCODED[label]
    if label not in _color_cache:
        _color_cache[label] = next(_fallback)
    return _color_cache[label]

def _style_ax(ax, title, xlabel, ylabel=None, ylim=None, xlim=None):
    ax.set_title(title, **title_font)
    if xlabel:
        ax.set_xlabel(xlabel, **axis_font)
    if ylabel:
        ax.set_ylabel(ylabel, **axis_font)
    ax.tick_params(axis='both', labelsize=axis_font['size'])
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight('bold')
    if ylim:
        ax.set_ylim(ylim)
    if xlim:
        ax.set_xlim(xlim)
    ax.grid(True, alpha=0.4)

print('Engine ready')

## 6 — ET1 / ET2: HSJA robustness, with vs without adversarial training (no FL)

Unchanged proposal, retargeted to the new groups `et2-noadv-nofl` / `et2-adv-nofl`. A 3-row x 6-col grid: columns are grouped 2-per-architecture (MLP | CNN | ResNet); rows are (Accuracy, Precision) / (Recall, Macro-F1) / (Avg L2 perturbation, Avg queries). Within each panel: 4 vehicles (colour) x 2 training regimes (dotted = no-adv, solid = adv).

#### Horizontal:

In [ ]:
def plot_hsja_architecture_grid(custom_ylims=None, xlim=None, smooth=2, intervals=False):
    custom_ylims = custom_ylims or {}
    metrics_to_plot = ['hsja_adv_eval/accuracy', 'hsja_adv_eval/precision',
                       'hsja_adv_eval/recall', 'hsja_adv_eval/macro_f1',
                       'hsja_adv_eval/avg_perturbation', 'hsja_adv_eval/avg_queries']
    metric_row = {metrics_to_plot[0]:0, metrics_to_plot[1]:0,
                  metrics_to_plot[2]:1, metrics_to_plot[3]:1,
                  metrics_to_plot[4]:2, metrics_to_plot[5]:2}
    metric_off = {metrics_to_plot[0]:0, metrics_to_plot[1]:1,
                  metrics_to_plot[2]:0, metrics_to_plot[3]:1,
                  metrics_to_plot[4]:0, metrics_to_plot[5]:1}
    arch_col_start = {'mlp':0, 'cnn':2, 'resnet':4}
    regimes = [(ET2_NOADV, 'no-adv-training', ':'), (ET2_ADV, 'adv-training', '-')]

    fig, axes = plt.subplots(3, 6, figsize=(6*5.5, 3*5), squeeze=False)
    ci_alpha = 0.95
    for arch in MODELS:
        base_col = arch_col_start[arch]
        for metric in metrics_to_plot:
            ax = axes[metric_row[metric]][base_col + metric_off[metric]]
            for v in VEHICLES:
                for base, _lab, ls in regimes:
                    wide = seed_series_for_group(with_arch(base, arch), metric_key(v, metric), smooth)
                    mean, lo, hi = compute_ci_band(wide, ci_alpha)
                    if mean is None:
                        continue
                    c = color_for(v.capitalize())
                    ax.plot(mean.index, mean.values, color=c, linestyle=ls, lw=2.4, alpha=0.95)
                    if intervals and lo is not None:
                        ax.fill_between(lo.index, lo.values, hi.values, color=c, alpha=0.16)
            title = '%s - %s' % (arch.upper(), pretty(metric))
            yl = custom_ylims.get(title, ylim_for(metric))
            _style_ax(ax, title, '', None, yl, xlim)

    handles = [Line2D([0],[0], color=color_for(v.capitalize()), lw=3.2, label=v.capitalize()) for v in VEHICLES]
    handles += [Line2D([0],[0], color='gray', linestyle=':', lw=3.2, label='no-adv-training'),
                Line2D([0],[0], color='gray', linestyle='-', lw=3.2, label='adv-training'),
                mpatches.Patch(alpha=0.3, color='grey', label='mean +/- 95%% CI (across seeds)')]
    fig.suptitle('HSJA adversarial evaluation with and without adversarial training (no FL)',
                 weight='bold', size=30, y=0.95)
    fig.supxlabel('<--- ADVERSARIAL EVALUATION ROUND --->', weight='bold', size=28, y=0.04)
    fig.legend(handles=handles, prop={'weight':'bold','size':24}, loc='lower center',
               bbox_to_anchor=(0.5, -0.02), ncol=7, frameon=True)
    plt.tight_layout(rect=[0, 0.04, 1, 0.96])
    plt.show()

plot_hsja_architecture_grid(
    custom_ylims={'%s - Avg L2 perturbation' % a.upper(): (0.0, 0.6) for a in MODELS} |
                 {'%s - Avg queries' % a.upper(): (0.0, 1000.0) for a in MODELS} |
                 {'%s - Accuracy' % a.upper(): (0.5, 1.02) for a in MODELS} |
                 {'%s - Precision' % a.upper(): (0.5, 1.02) for a in MODELS} |
                 {'%s - Recall' % a.upper(): (0.5, 1.02) for a in MODELS} |
                 {'%s - Macro-F1' % a.upper(): (0.5, 1.02) for a in MODELS},
    xlim=(0, 30)
    )

#### Horizontal (clean):

In [ ]:
import itertools
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

title_font  = {'weight': 'bold', 'size': 22}
axis_font   = {'weight': 'bold', 'size': 18}
legend_font = {'weight': 'bold', 'size': 16}

_HARDCODED = {
    'no-adv-training': '#E8743B', 'adv-training': '#19A979',
    'FL off': '#B06B5A', 'FL on': '#2C7FB8',
    'FedAvg': '#3F51B5', 'FedProx': '#009688', 'FedYogi': '#E91E63', 'FedMedian': '#FF9800',
    'mlp': '#2196F3', 'cnn': '#FF9800', 'resnet': '#4CAF50',
    'Angela': '#2E86AB', 'Bob': '#19A979', 'Claude': '#E8743B', 'Daniel': '#A23B72',
    'angela': '#2E86AB', 'bob': '#19A979', 'claude': '#E8743B', 'daniel': '#A23B72',
    'peers': '#19A979',
}
_fallback = itertools.cycle(plt.colormaps.get_cmap('tab10').colors)
_color_cache = {}
def color_for(label):
    if label in _HARDCODED:
        return _HARDCODED[label]
    if label not in _color_cache:
        _color_cache[label] = next(_fallback)
    return _color_cache[label]

def _style_ax(ax, title, xlabel, ylabel=None, ylim=None, xlim=None, show_yticklabels=True, show_xticklabels=True):
    ax.set_title(title, **title_font)
    if xlabel:
        ax.set_xlabel(xlabel, **axis_font)
    if ylabel:
        ax.set_ylabel(ylabel, **axis_font)

    # Increase tick font size
    ax.tick_params(axis='both', labelsize=20)

    # Conditionally remove y-axis ticks and labels
    if not show_yticklabels:
        ax.tick_params(axis='y', labelleft=False, left=False)

    # Conditionally remove x-axis ticks and labels
    if not show_xticklabels:
        ax.tick_params(axis='x', labelbottom=False, bottom=False)

    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontweight('bold')
    if ylim:
        ax.set_ylim(ylim)
    if xlim:
        ax.set_xlim(xlim)
    ax.grid(True, alpha=0.4)

def plot_hsja_architecture_grid(custom_ylims=None, xlim=None, smooth=2, intervals=False):
    custom_ylims = custom_ylims or {}
    metrics_to_plot = ['hsja_adv_eval/accuracy', 'hsja_adv_eval/precision',
                       'hsja_adv_eval/recall', 'hsja_adv_eval/macro_f1',
                       'hsja_adv_eval/avg_perturbation', 'hsja_adv_eval/avg_queries']
    metric_row = {metrics_to_plot[0]:0, metrics_to_plot[1]:0,
                  metrics_to_plot[2]:1, metrics_to_plot[3]:1,
                  metrics_to_plot[4]:2, metrics_to_plot[5]:2}
    metric_off = {metrics_to_plot[0]:0, metrics_to_plot[1]:1,
                  metrics_to_plot[2]:0, metrics_to_plot[3]:1,
                  metrics_to_plot[4]:0, metrics_to_plot[5]:1}
    arch_col_start = {'mlp':0, 'cnn':2, 'resnet':4}
    regimes = [(ET2_NOADV, 'no-adv-training', ':'), (ET2_ADV, 'adv-training', '-')]

    total_rows = 3 # Hardcoded from fig, axes = plt.subplots(3, 6, ...)

    fig, axes = plt.subplots(total_rows, 6, figsize=(6*5.5, total_rows*5), squeeze=False)
    ci_alpha = 0.95
    for arch in MODELS:
        base_col = arch_col_start[arch]
        for metric in metrics_to_plot:
            current_row = metric_row[metric]
            ax = axes[current_row][base_col + metric_off[metric]]
            for v in VEHICLES:
                for base, _lab, ls in regimes:
                    wide = seed_series_for_group(with_arch(base, arch), metric_key(v, metric), smooth)
                    mean, lo, hi = compute_ci_band(wide, ci_alpha)
                    if mean is None:
                        continue
                    c = color_for(v.capitalize())
                    ax.plot(mean.index, mean.values, color=c, linestyle=ls, lw=2.4, alpha=0.95)
                    if intervals and lo is not None:
                        ax.fill_between(lo.index, lo.values, hi.values, color=c, alpha=0.16)
            title = '%s - %s' % (arch.upper(), pretty(metric))
            yl = custom_ylims.get(title, ylim_for(metric))
            # Pass show_yticklabels=True only for the first column
            # Pass show_xticklabels=True only for the last row
            _style_ax(ax, title, '', None, yl, xlim,
                      show_yticklabels=(base_col + metric_off[metric] == 0),
                      show_xticklabels=(current_row == total_rows - 1))

    handles = [Line2D([0],[0], color=color_for(v.capitalize()), lw=3.2, label=v.capitalize()) for v in VEHICLES]
    handles += [Line2D([0],[0], color='gray', linestyle=':', lw=3.2, label='no-adv-training'),
                Line2D([0],[0], color='gray', linestyle='-', lw=3.2, label='adv-training'),
                mpatches.Patch(alpha=0.3, color='grey', label='mean +/- 95%% CI (across seeds)')]
    fig.suptitle('HSJA adversarial evaluation with and without adversarial training (no FL)',
                 weight='bold', size=30, y=0.95)
    fig.supxlabel('<--- ADVERSARIAL EVALUATION ROUND --->', weight='bold', size=28, y=0.04)
    fig.legend(handles=handles, prop={'weight':'bold','size':24}, loc='lower center',
               bbox_to_anchor=(0.5, -0.02), ncol=7, frameon=True)
    plt.tight_layout(rect=[0, 0.04, 1, 0.96])
    plt.show()

plot_hsja_architecture_grid(
    custom_ylims={'%s - Avg L2 perturbation' % a.upper(): (0.0, 0.5) for a in MODELS} |
                 {'%s - Avg queries' % a.upper(): (0.0, 1000.0) for a in MODELS} |
                 {'%s - Accuracy' % a.upper(): (0.5, 1.02) for a in MODELS} |
                 {'%s - Precision' % a.upper(): (0.5, 1.02) for a in MODELS} |
                 {'%s - Recall' % a.upper(): (0.5, 1.02) for a in MODELS} |
                 {'%s - Macro-F1' % a.upper(): (0.5, 1.02) for a in MODELS},
    xlim=(0, 30)
    )

#### Vertical:

In [ ]:
title_font  = {'weight': 'bold', 'size': 22}
axis_font   = {'weight': 'bold', 'size': 22}
legend_font = {'weight': 'bold', 'size': 20}


def plot_hsja_architecture_grid(custom_ylims=None, xlim=None, smooth=2, intervals=False):
    custom_ylims = custom_ylims or {}
    metrics_to_plot = ['hsja_adv_eval/accuracy', 'hsja_adv_eval/precision',
                       'hsja_adv_eval/recall', 'hsja_adv_eval/macro_f1',
                       'hsja_adv_eval/avg_perturbation', 'hsja_adv_eval/avg_queries']
    regimes = [(ET2_NOADV, 'no-adv-training', ':'), (ET2_ADV, 'adv-training', '-')]

    num_metric_rows = len(metrics_to_plot) # 6 rows
    num_arch_cols = len(MODELS)            # 3 columns

    # Adjusted figsize for new dimensions (6 rows x 3 cols)
    fig, axes = plt.subplots(num_metric_rows, num_arch_cols,
                             figsize=(num_arch_cols * 6, num_metric_rows * 4), 
                             squeeze=False)
    ci_alpha = 0.95

    # Iterate metrics for rows, architectures for columns
    for r_idx, metric in enumerate(metrics_to_plot):
        for c_idx, arch in enumerate(MODELS):
            ax = axes[r_idx][c_idx]

            for v in VEHICLES:
                for base, _lab, ls in regimes:
                    wide = seed_series_for_group(with_arch(base, arch), metric_key(v, metric), smooth)
                    mean, lo, hi = compute_ci_band(wide, ci_alpha)
                    if mean is None:
                        continue
                    c = color_for(v.capitalize())
                    ax.plot(mean.index, mean.values, color=c, linestyle=ls, lw=2.4, alpha=0.95)
                    if intervals and lo is not None:
                        ax.fill_between(lo.index, lo.values, hi.values, color=c, alpha=0.16)

            # Calculate title for ylim lookup
            title_key = '%s - %s' % (arch.upper(), pretty(metric))
            yl = custom_ylims.get(title_key, ylim_for(metric))
            
            # Apply base styling (grid, limits, tick params size) but avoid setting individual titles/labels
            _style_ax(ax, '', '', None, yl, xlim)

            # Manually control tick label visibility
            # Only show y-axis labels for the first column
            if c_idx == 0:
                ax.set_ylabel(pretty(metric), **axis_font)
                ax.tick_params(axis='y', labelleft=True) # Ensure labels are visible
            else:
                ax.tick_params(axis='y', labelleft=False)

            # Only show x-axis labels for the last row 
            if r_idx == num_metric_rows - 1:

              ax.tick_params(axis='x', labelbottom=True) # Ensure labels are visible
              if c_idx == 1: # only mid col
                ax.set_xlabel('<--- ADVERSARIAL EVALUATION ROUND --->', **axis_font)
                
            else:
                ax.tick_params(axis='x', labelbottom=False)

            # Set architecture title above each column (top row only)
            if r_idx == 0:
                ax.set_title(arch.upper(), **title_font)
    
    # Legend setup
    handles = [Line2D([0],[0], color=color_for(v.capitalize()), lw=3.2, label=v.capitalize()) for v in VEHICLES]
    handles += [Line2D([0],[0], color='gray', linestyle=':', lw=3.2, label='no-adv-training'),
                Line2D([0],[0], color='gray', linestyle='-', lw=3.2, label='adv-training'),
                mpatches.Patch(alpha=0.3, color='grey', label='mean +/- 95%% CI (across seeds)')]

    # Overall suptitle for the plot
    fig.suptitle('HSJA adversarial evaluation with and without adversarial training (no FL)',
                 weight='bold', size=28, y=0.96, x=0.54) # Adjusted y for overall title
    
    fig.legend(handles=handles, prop={'weight':'bold','size':24}, loc='lower center',
               bbox_to_anchor=(0.55, -0.02), ncol=3, frameon=True)
    
    # Adjust tight_layout rect to accommodate new title and row labels
    plt.tight_layout(rect=[0.05, 0.05, 1, 0.96]) # Increased left padding for ylabels, adjusted top for suptitle
    plt.show()

plot_hsja_architecture_grid(
    custom_ylims={'%s - Avg L2 perturbation' % a.upper(): (0.0, 0.8) for a in MODELS} |
                 {'%s - Avg queries' % a.upper(): (0.0, 800.0) for a in MODELS} |
                 {'%s - Accuracy' % a.upper(): (0.6, 1.02) for a in MODELS} |
                 {'%s - Precision' % a.upper(): (0.6, 1.02) for a in MODELS} |
                 {'%s - Recall' % a.upper(): (0.6, 1.02) for a in MODELS} |
                 {'%s - Macro-F1' % a.upper(): (0.6, 1.02) for a in MODELS},
    xlim=(0, 30)
    )

## 7 — ET3: Federated robustness transfer to the clean free-rider (Block B)

Setup: Bob/Claude/Daniel train adversarially at a **common** noise level; **Angela is the clean free-rider** (`adversarial_training=False`). Evaluation is identical for all vehicles (clean-anchor HSJA + the sigma-grid). The **only** independent variable is FL on/off (+ aggregation strategy). Transfer is demonstrated if **Angela's** robustness rises toward the adversarially-trained peers *only when FL is on*.

### 7.1 — Fig A: FL off vs FL on, per architecture x aggregator (converged HSJA)

In [ ]:
def et3_offon_bars(metric='hsja_adv_eval/avg_perturbation', tail=5, smooth=2):
    aggs = list(ET3_ON)
    fig, axes = plt.subplots(len(MODELS), len(aggs),
                             figsize=(5.4*len(aggs), 4.8*len(MODELS)),
                             squeeze=False, sharey='row')
    x = np.arange(len(VEHICLES)); w = 0.38
    for r, arch in enumerate(MODELS):
        for c, agg in enumerate(aggs):
            ax = axes[r][c]
            off_m, off_e, on_m, on_e = [], [], [], []
            for v in VEHICLES:
                m0, e0, _ = converged_scalar(with_arch(ET3_OFF, arch), metric_key(v, metric), tail, smooth)
                m1, e1, _ = converged_scalar(with_arch(ET3_ON[agg], arch), metric_key(v, metric), tail, smooth)
                off_m.append(m0); off_e.append(e0 or 0.0); on_m.append(m1); on_e.append(e1 or 0.0)
            ax.bar(x - w/2, off_m, w, yerr=off_e, capsize=3, color=color_for('FL off'),
                   edgecolor='black', label='FL off')
            ax.bar(x + w/2, on_m,  w, yerr=on_e,  capsize=3, color=color_for('FL on'),
                   edgecolor='black', label='FL on')
            # mark the free-rider column
            fr = VEHICLES.index(FREE_RIDER)
            ax.axvspan(fr - 0.5, fr + 0.5, color='gold', alpha=0.12, zorder=0)
            _style_ax(ax, '%s | %s' % (arch.upper(), agg), '',
                      pretty(metric) if c == 0 else None, ylim_for(metric))
            ax.set_xticks(x); ax.set_xticklabels([v.capitalize() for v in VEHICLES],
                                                 fontweight='bold', fontsize=14)
    handles = [Patch(facecolor=color_for('FL off'), edgecolor='black', label='FL off'),
               Patch(facecolor=color_for('FL on'), edgecolor='black', label='FL on'),
               Patch(facecolor='gold', alpha=0.3, label='free-rider (Angela)')]
    fig.legend(handles=handles, prop={'weight':'bold','size':20}, loc='lower center',
               bbox_to_anchor=(0.5, -0.01), ncol=3, frameon=True)
    fig.suptitle('ET3 - %s: FL off vs FL on (rows: architecture, cols: aggregator)' % pretty(metric),
                 weight='bold', size=26, y=0.97)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

et3_offon_bars('hsja_adv_eval/avg_perturbation')
et3_offon_bars('hsja_adv_eval/accuracy')

### 7.2 — Fig B: Angela-centric transfer (does the free-rider reach the peer band?)

For each architecture, Angela's converged robustness across `{FL off, FedAvg, FedProx, FedYogi, FedMedian}`. The green band is the adversarially-trained peers' mean +/- std (under FL on) — the 'target' Angela transfers toward. If FL works, Angela's FL-on bars climb from the grey FL-off baseline into the band.

In [ ]:
def et3_angela_transfer(metric='hsja_adv_eval/avg_perturbation', tail=5, smooth=2):
    cats = ['FL off'] + list(ET3_ON)
    groups = [ET3_OFF] + [ET3_ON[a] for a in ET3_ON]
    fig, axes = plt.subplots(1, len(MODELS), figsize=(6.6*len(MODELS), 5.4), squeeze=False)
    for i, arch in enumerate(MODELS):
        ax = axes[0][i]
        vals, errs = [], []
        for g in groups:
            m, e, _ = converged_scalar(with_arch(g, arch), metric_key(FREE_RIDER, metric), tail, smooth)
            vals.append(m); errs.append(e or 0.0)
        xx = np.arange(len(cats))
        colors = [color_for('FL off')] + [color_for('FL on')]*len(ET3_ON)
        ax.bar(xx, vals, 0.62, yerr=errs, capsize=4, color=colors, edgecolor='black')
        # peer reference band (adv-trained peers under FL on, across aggregators)
        peer_vals = []
        for a in ET3_ON:
            for p in PEERS:
                m, _, _ = converged_scalar(with_arch(ET3_ON[a], arch), metric_key(p, metric), tail, smooth)
                if not np.isnan(m):
                    peer_vals.append(m)
        if peer_vals:
            pm, ps = float(np.mean(peer_vals)), float(np.std(peer_vals))
            ax.axhspan(pm - ps, pm + ps, color=color_for('peers'), alpha=0.15, zorder=0)
            ax.axhline(pm, color=color_for('peers'), lw=2.4, ls='--', label='adv-trained peers (FL on)')
        _style_ax(ax, arch.upper(), '', ('Angela - %s' % pretty(metric)) if i == 0 else None, ylim_for(metric))
        ax.set_xticks(xx); ax.set_xticklabels(cats, rotation=20, ha='right', fontweight='bold', fontsize=13)
    handles = [Patch(facecolor=color_for('FL off'), edgecolor='black', label='Angela (FL off)'),
               Patch(facecolor=color_for('FL on'), edgecolor='black', label='Angela (FL on)'),
               Line2D([0],[0], color=color_for('peers'), lw=2.8, ls='--', label='adv-trained peers (FL on)')]
    fig.legend(handles=handles, prop={'weight':'bold','size':18}, loc='lower center',
               bbox_to_anchor=(0.5, -0.04), ncol=3, frameon=True)
    fig.suptitle('ET3 - free-rider transfer: Angela vs adversarially-trained peers (%s)' % pretty(metric),
                 weight='bold', size=24, y=1.0)
    plt.tight_layout(rect=[0, 0.02, 1, 0.96])
    plt.show()

et3_angela_transfer('hsja_adv_eval/avg_perturbation')
et3_angela_transfer('hsja_adv_eval/accuracy')

### 7.3 — Fig C: transfer dynamics (time-series)

HSJA robustness over benchmark rounds, all four vehicles (colour), FL off (dotted) vs FL on (solid), rows = architecture, cols = aggregator. Shows *when* Angela's curve lifts toward the peers once aggregation begins.

In [ ]:
def et3_transfer_dynamics(metric='hsja_adv_eval/avg_perturbation', smooth=2):
    aggs = list(ET3_ON)
    fig, axes = plt.subplots(len(MODELS), len(aggs),
                             figsize=(5.2*len(aggs), 4.3*len(MODELS)), squeeze=False)
    for r, arch in enumerate(MODELS):
        for c, agg in enumerate(aggs):
            ax = axes[r][c]
            for v in VEHICLES:
                for grp, ls in [(ET3_OFF, ':'), (ET3_ON[agg], '-')]:
                    wide = seed_series_for_group(with_arch(grp, arch), metric_key(v, metric), smooth)
                    mean, lo, hi = compute_ci_band(wide)
                    if mean is None:
                        continue
                    c_ = color_for(v.capitalize())
                    ax.plot(mean.index, mean.values, color=c_, linestyle=ls, lw=2.2, alpha=0.95)
                    if lo is not None:
                        ax.fill_between(lo.index, lo.values, hi.values, color=c_, alpha=0.10)
            _style_ax(ax, '%s | %s' % (arch.upper(), agg), 'round',
                      pretty(metric) if c == 0 else None, ylim_for(metric))
    handles = [Line2D([0],[0], color=color_for(v.capitalize()), lw=3.0, label=v.capitalize()) for v in VEHICLES]
    handles += [Line2D([0],[0], color='gray', ls=':', lw=3.0, label='FL off'),
                Line2D([0],[0], color='gray', ls='-', lw=3.0, label='FL on')]
    fig.legend(handles=handles, prop={'weight':'bold','size':18}, loc='lower center',
               bbox_to_anchor=(0.5, -0.02), ncol=6, frameon=True)
    fig.suptitle('ET3 - transfer dynamics: %s over rounds' % pretty(metric), weight='bold', size=24, y=0.99)
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.show()

et3_transfer_dynamics('hsja_adv_eval/avg_perturbation')

### 7.4 — Aggregator comparison: which strategy transfers best to the free-rider?

For the clean free-rider (Angela), converged robustness under `FL off` and each aggregator `{FedAvg, FedProx, FedYogi, FedMedian}`, grouped by architecture. The taller the FL-on bars sit above the grey `FL off` bar, the stronger the transfer that aggregator delivers.

In [ ]:
def et3_aggregator_comparison(metric='hsja_adv_eval/avg_perturbation', who=FREE_RIDER,
                              tail=5, smooth=2):
    cats   = ['FL off'] + list(ET3_ON)
    groups = [ET3_OFF] + [ET3_ON[a] for a in ET3_ON]
    x = np.arange(len(MODELS)); w = 0.8 / len(cats)
    fig, ax = plt.subplots(figsize=(max(11, 3.4*len(MODELS)), 6))
    for j, (cat, g) in enumerate(zip(cats, groups)):
        means, errs = [], []
        for arch in MODELS:
            m, e, _ = converged_scalar(with_arch(g, arch), metric_key(who, metric), tail, smooth)
            means.append(m); errs.append(e or 0.0)
        ax.bar(x + j*w - 0.4 + w/2, means, w, yerr=errs, capsize=3, label=cat,
               color=color_for(cat), edgecolor='black', linewidth=1.0)
    ax.set_xticks(x); ax.set_xticklabels([m.upper() for m in MODELS], fontweight='bold', fontsize=15)
    _style_ax(ax, '%s transfer to %s by aggregator' % (pretty(metric), who.capitalize()),
              'Architecture', pretty(metric), ylim_for(metric))
    ax.legend(prop=legend_font, ncol=len(cats))
    plt.tight_layout(); plt.show()

et3_aggregator_comparison('hsja_adv_eval/avg_perturbation')
et3_aggregator_comparison('hsja_adv_eval/accuracy')

### 7.5 — ET3 converged-summary table (transfer gain Δ)

Per architecture × vehicle: the converged `FL off` value, each aggregator's `FL on` value (mean ± 95% CI across seeds), and the **transfer gain Δ = FL on − FL off**. Angela's Δ is the headline transfer number; the peers should already be high with small Δ.

In [ ]:
from IPython.display import display, Markdown

def _fmt(m, e):
    if np.isnan(m):
        return 'N/A'
    return '%.3f +/- %.3f' % (m, e) if e else '%.3f' % m

def et3_summary_table(metric='hsja_adv_eval/avg_perturbation', tail=5, smooth=2):
    rows = []
    for arch in MODELS:
        for v in VEHICLES:
            off_m, off_e, _ = converged_scalar(with_arch(ET3_OFF, arch), metric_key(v, metric), tail, smooth)
            row = {'Architecture': arch.upper(),
                   'Vehicle': v.capitalize() + ('*' if v == FREE_RIDER else ''),
                   'FL off': _fmt(off_m, off_e)}
            for agg, g in ET3_ON.items():
                on_m, on_e, _ = converged_scalar(with_arch(g, arch), metric_key(v, metric), tail, smooth)
                row[agg] = _fmt(on_m, on_e)
                d = (on_m - off_m) if (not np.isnan(on_m) and not np.isnan(off_m)) else np.nan
                row['%s d' % agg] = ('%+.3f' % d) if not np.isnan(d) else 'N/A'
            rows.append(row)
    df = pd.DataFrame(rows)
    display(Markdown('**ET3 converged %s  -  transfer gain d = (FL on) - (FL off).  '
                     '* = clean free-rider.**' % pretty(metric)))
    display(Markdown(df.to_markdown(index=False)))
    return df

_ = et3_summary_table('hsja_adv_eval/avg_perturbation')
_ = et3_summary_table('hsja_adv_eval/accuracy')

## 8 — ET4: in-distribution cost vs robustness (sigma-degradation curves)

Exploiting the new fixed sigma-grid. For each architecture (rows) x vehicle (cols), accuracy (or macro-F1) as a function of evaluation noise `sigma`, for no-adv (dotted) vs adv (solid) training. The horizontal reference lines are the in-distribution (clean, balanced training) performance per regime. The reading: adversarial training trades a little in-distribution accuracy for a **slower degradation** as sigma grows (flatter solid curve).

In [ ]:
def et4_sigma_degradation(metric_base='accuracy', tail=5, smooth=2, show_indist=True):
    regimes = [(ET2_NOADV, 'no-adv-training', ':'), (ET2_ADV, 'adv-training', '-')]
    indist_key = {'accuracy':'class_accuracy', 'macro_f1':'class_macro_f1',
                  'f1':'class_f1'}.get(metric_base, 'class_accuracy')
    fig, axes = plt.subplots(len(MODELS), len(VEHICLES),
                             figsize=(4.9*len(VEHICLES), 4.2*len(MODELS)),
                             squeeze=False, sharey=True)
    for r, arch in enumerate(MODELS):
        for c, v in enumerate(VEHICLES):
            ax = axes[r][c]
            for base, lab, ls in regimes:
                g = with_arch(base, arch)
                col = color_for(lab)
                xs, ys, es = [], [], []
                for s in EVAL_SIGMAS:
                    m, e, _ = converged_scalar(g, metric_key(v, sigma_key(metric_base, s)), tail, smooth)
                    if not np.isnan(m):
                        xs.append(s); ys.append(m); es.append(e or 0.0)
                if xs:
                    ax.errorbar(xs, ys, yerr=es, color=col, linestyle=ls, lw=2.6,
                                marker='o', markersize=6, capsize=3)
                if show_indist:
                    m, _, _ = converged_scalar(g, metric_key(v, indist_key), tail, smooth)
                    if not np.isnan(m):
                        ax.axhline(m, color=col, ls=ls, lw=1.1, alpha=0.5)
            title = '%s | %s (sigma_tr=%s)' % (arch.upper(), v.capitalize(), VEHICLE_NOISE[v])
            _style_ax(ax, title, 'evaluation sigma',
                      ('%s' % pretty('hsja_adv_eval/%s' % metric_base)) if c == 0 else None, (0, 1.02))
    handles = [Line2D([0],[0], color=color_for('no-adv-training'), ls=':', lw=3.0, marker='o', label='no-adv-training'),
               Line2D([0],[0], color=color_for('adv-training'), ls='-', lw=3.0, marker='o', label='adv-training'),
               Line2D([0],[0], color='gray', ls='-', lw=1.2, alpha=0.6, label='in-distribution reference')]
    fig.legend(handles=handles, prop={'weight':'bold','size':18}, loc='lower center',
               bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=True)
    fig.suptitle('ET4 - Gaussian sigma-degradation: %s vs evaluation noise' % metric_base,
                 weight='bold', size=24, y=0.99)
    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.show()

et4_sigma_degradation('accuracy')
et4_sigma_degradation('macro_f1')

## 9 — Fig 5: representation-space grid (Plotly artifacts)

Ported from the previous notebook (2nd-submission layout), retargeted to the new no-FL groups `et2-noadv-nofl` / `et2-adv-nofl`. Rows: Angela (no-adv), Daniel (no-adv), Daniel (adv); columns: input-space PCA + 2D representation (labels/preds) across MLP / CNN / ResNet.

In [ ]:
import tempfile, json as _json, base64, shutil
import plotly, plotly.graph_objects as go
from plotly.subplots import make_subplots

PLOT_SEED   = 42
MARKER_SIZE = 15
TICK_SIZE, AXIS_TITLE_SIZE, TITLE_SIZE, LEGEND_SIZE = 18, 30, 42, 36
FONT_FAMILY = 'DejaVu Sans, Arial, sans-serif'
SAVE_PNG    = False
_PLOTLY_WEIGHT = tuple(int(x) for x in plotly.__version__.split('.')[:2]) >= (5, 22)
def _bf(size, bold=True):
    d = {'size': size, 'family': FONT_FAMILY}
    if bold and _PLOTLY_WEIGHT:
        d['weight'] = 'bold'
    return d

def find_run(group, seed):
    api = wandb.Api(timeout=60)
    runs = list(api.runs(WANDB_PROJECT, filters={'group': group}))
    cands = [r for r in runs if _seed_of(r) == seed]
    if WANDB_TAG_FILTER:
        tagged = [r for r in cands if any(t in r.tags for t in WANDB_TAG_FILTER)]
        cands = tagged or cands
    if not cands:
        return None
    cands.sort(key=lambda r: (r.summary.get('_step', 0) or 0), reverse=True)
    return cands[0]

def _decode_bdata(obj):
    if isinstance(obj, dict):
        if 'bdata' in obj and 'dtype' in obj:
            arr = np.frombuffer(base64.b64decode(obj['bdata']), dtype=np.dtype(obj['dtype']))
            shape = obj.get('shape')
            if shape:
                arr = arr.reshape([int(s) for s in str(shape).split(',')])
            return arr
        return {k: _decode_bdata(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_decode_bdata(v) for v in obj]
    return obj

def _load_plotly_file(path):
    raw = _json.loads(Path(path).read_text())
    d = _decode_bdata(raw)
    return go.Figure(data=d.get('data'), layout=d.get('layout'))

_PLOTLY_FILE_CACHE = {}
def _plotly_files(run):
    if run.id not in _PLOTLY_FILE_CACHE:
        _PLOTLY_FILE_CACHE[run.id] = [f for f in run.files()
            if f.name.startswith('media/plotly/') and f.name.endswith('.json')]
    return _PLOTLY_FILE_CACHE[run.id]

def download_last_plotly(run, key, dest):
    prefix = 'media/plotly/%s_' % key
    matches = [f for f in _plotly_files(run) if f.name.startswith(prefix)]
    if not matches:
        return None
    def _step(f):
        tail = f.name[len(prefix):]
        try:
            return int(tail.split('_')[0])
        except Exception:
            return -1
    f = max(matches, key=_step)
    f.download(root=str(dest), replace=True)
    return _load_plotly_file(Path(dest) / f.name)

print('Plotly helpers ready')

In [ ]:
def plot_manifold_grid():
    rows_data = [
        {'vehicle': 'angela', 'group': ET2_NOADV, 'row_title': 'Angela - no adversarial training'},
        {'vehicle': 'daniel', 'group': ET2_NOADV, 'row_title': 'Daniel - no adversarial training'},
        {'vehicle': 'daniel', 'group': ET2_ADV,   'row_title': 'Daniel - adversarial training'},
    ]
    column_configs = [
        {'arch':'mlp',    'suffix':'manifold_pca',    'col_title':'Input-Space PCA (MLP)'},
        {'arch':'mlp',    'suffix':'manifold_labels', 'col_title':'2D-Repr. labels (MLP)'},
        {'arch':'mlp',    'suffix':'manifold_preds',  'col_title':'2D-Repr. preds (MLP)'},
        {'arch':'cnn',    'suffix':'manifold_labels', 'col_title':'2D-Repr. labels (CNN)'},
        {'arch':'cnn',    'suffix':'manifold_preds',  'col_title':'2D-Repr. preds (CNN)'},
        {'arch':'resnet', 'suffix':'manifold_labels', 'col_title':'2D-Repr. labels (ResNet)'},
        {'arch':'resnet', 'suffix':'manifold_preds',  'col_title':'2D-Repr. preds (ResNet)'},
    ]
    dest = Path(tempfile.mkdtemp(prefix='serebench2_plotly_'))
    fig = make_subplots(rows=len(rows_data), cols=len(column_configs),
                        horizontal_spacing=0.02, vertical_spacing=0.10)
    axis_kw = dict(tickfont=_bf(TICK_SIZE), title_font=_bf(AXIS_TITLE_SIZE),
                   showline=True, linewidth=1.8, linecolor='black',
                   ticks='outside', tickwidth=1.8, ticklen=6,
                   showgrid=True, gridcolor='rgba(0,0,0,0.12)')
    all_traces = []
    for r, rc in enumerate(rows_data):
        for c, cc in enumerate(column_configs):
            grp = with_arch(rc['group'], cc['arch'])
            run = find_run(grp, PLOT_SEED)
            xt, yt = ('PCA_1','PCA_2') if cc['suffix'] == 'manifold_pca' else ('DIM_1','DIM_2')
            fig.update_xaxes(title_text=xt, row=r+1, col=c+1, **axis_kw)
            fig.update_yaxes(title_text=yt, row=r+1, col=c+1, **axis_kw)
            if run is None:
                print('  ! no run for %s seed=%s' % (grp, PLOT_SEED))
                fig.add_trace(go.Scatter(mode='markers', x=[None], y=[None], showlegend=False), row=r+1, col=c+1)
                continue
            try:
                fig_data = download_last_plotly(run, '%s_%s' % (rc['vehicle'], cc['suffix']), dest)
            except Exception as exc:
                print('  ! download failed for %s/%s: %s' % (grp, cc['suffix'], exc)); fig_data = None
            if fig_data is None:
                fig.add_trace(go.Scatter(mode='markers', x=[None], y=[None], showlegend=False), row=r+1, col=c+1)
                continue
            for trace in fig_data.data:
                trace.marker.size = MARKER_SIZE
                all_traces.append({'trace': trace, 'row': r+1, 'col': c+1, 'suffix': cc['suffix']})
    seen = set()
    for ti in all_traces:
        tr = ti['trace']
        if ti['suffix'] == 'manifold_labels' and tr.name:
            tr.showlegend = tr.name not in seen; seen.add(tr.name); tr.legendgroup = tr.name
        else:
            tr.showlegend = False
        fig.add_trace(tr, row=ti['row'], col=ti['col'])
    ann = [dict(text='Representation-space grid across vehicles, training regimes and architectures',
                xref='paper', yref='paper', x=0.5, y=1.06, xanchor='center', yanchor='bottom',
                showarrow=False, font=dict(size=TITLE_SIZE, weight='bold'))]
    for c, cc in enumerate(column_configs):
        ann.append(dict(text=cc['col_title'], xref='paper', yref='paper', y=1.01,
                        x=(c+0.5)/len(column_configs), xanchor='center', yanchor='bottom',
                        showarrow=False, font=_bf(AXIS_TITLE_SIZE)))
    for r, rc in enumerate(rows_data):
        ann.append(dict(text=rc['row_title'], xref='paper', yref='paper', x=-0.01,
                        y=1 - (r+0.5)/len(rows_data), xanchor='right', yanchor='middle',
                        showarrow=False, font=_bf(AXIS_TITLE_SIZE), textangle=-90))
    fig.update_layout(width=680*len(column_configs), height=640*len(rows_data),
                      template='plotly_white', font=dict(size=TICK_SIZE, family=FONT_FAMILY),
                      margin=dict(l=220, r=40, t=200, b=200), showlegend=True,
                      legend=dict(font=_bf(LEGEND_SIZE), orientation='h', yanchor='bottom',
                                  y=-0.12, xanchor='center', x=0.5, borderwidth=1),
                      annotations=ann)
    fig.show()
    if SAVE_PNG:
        try:
            fig.write_image(str(dest / 'manifold_grid.png'), scale=2)
            print('Saved', dest / 'manifold_grid.png')
        except Exception as exc:
            print('   (PNG export skipped - install kaleido:', exc, ')')
    shutil.rmtree(dest, ignore_errors=True)

plot_manifold_grid()